# SpatialFlow End-to-End Runner

This notebook runs the core SpatialFlow pipeline:

1. Load config and data
2. Train `EAESpatialFlow`
3. Validate and run inference
4. Save outputs
5. Run optional attribution on PLP remission sections


In [1]:
# Optional one-time dependency install inside this notebook kernel.
# Set to True if imports fail.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[full]"])


In [2]:
from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "spatialflow").exists() and (ROOT.parent / "spatialflow").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

ROOT, DEVICE


(PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow'),
 'mps')

## Run Mode

- If `DATA_DIR` points to a real Xenium dataset arranged like `scripts/train.py` expects, that data is used.
- Otherwise, the notebook generates synthetic sections so the pipeline can run end-to-end.


In [3]:
# Set to your Xenium root when available, e.g. ROOT / 'data' / 'xenium'
DATA_DIR = None

USE_SYNTHETIC_IF_MISSING = True
OUTPUT_DIR = ROOT / "outputs" / "notebook_run"
N_EPOCHS_OVERRIDE = 2
RUN_ATTRIBUTE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow/outputs/notebook_run')

In [4]:
from spatialflow.aligner import AnatomicalAligner

def _synthetic_masks(h=120, w=120):
    tissue = np.ones((h, w), dtype=np.uint8)
    cc = np.zeros((h, w), dtype=np.uint8)
    gm = np.zeros((h, w), dtype=np.uint8)

    cy, cx = h // 2, w // 2
    cc[cy - 2:cy + 2, cx - 2:cx + 2] = 1

    yy, xx = np.ogrid[:h, :w]
    gm_ellipse = ((yy - cy) ** 2) / (0.25 * h * h) + ((xx - cx) ** 2) / (0.16 * w * w) <= 1.0
    gm[gm_ellipse] = 1
    gm[cc > 0] = 0

    return tissue, cc, gm

def _synthetic_section(model_type, timepoint, section_id, n_cells=180, n_genes=120):
    tissue, cc, gm = _synthetic_masks()

    xy = np.random.uniform(5, 115, (n_cells, 2)).astype(np.float32)

    lr_genes = [
        "cxcl10", "cxcr3", "ccl2", "ccr2", "tgfb1", "tgfbr1",
        "il1b", "il1r1", "tnf", "tnfrsf1a", "ifng", "ifngr1",
        "csf1", "csf1r", "vegfa", "kdr", "spp1", "cd44", "c3", "itgam"
    ]
    extra = [f"gene_{i:03d}" for i in range(max(0, n_genes - len(lr_genes)))]
    gene_names = (lr_genes + extra)[:n_genes]

    expr = np.abs(np.random.normal(loc=0.7, scale=0.5, size=(n_cells, len(gene_names)))).astype(np.float32)

    if model_type == "PLP" and "peak" in timepoint.lower():
        gene_to_idx = {g: i for i, g in enumerate(gene_names)}
        for g in ("cxcl10", "ccl2", "il1b", "tnf"):
            if g in gene_to_idx:
                expr[:, gene_to_idx[g]] += 0.8

    ct_pool = ["microglia", "astrocyte", "oligodendrocyte", "t_cell", "neuron"]
    cell_types = [ct_pool[i % len(ct_pool)] for i in range(n_cells)]

    return {
        "section_id": section_id,
        "model_type": model_type,
        "timepoint": timepoint,
        "xy_coords": xy,
        "expr_matrix": expr,
        "gene_names": gene_names,
        "cell_types": cell_types,
        "tissue_mask": tissue,
        "cc_mask": cc,
        "gm_mask": gm,
    }

def make_synthetic_sections():
    return [
        _synthetic_section("MOG", "baseline", "MOG_baseline_synth01"),
        _synthetic_section("MOG", "chronic_long", "MOG_chronic_long_synth01"),
        _synthetic_section("PLP", "remission_I", "PLP_remission_I_synth01"),
        _synthetic_section("PLP", "peak_II", "PLP_peak_II_synth01"),
    ]


/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from scripts.train import load_all_sections

with open(ROOT / "configs" / "default_config.yaml") as f:
    config = yaml.safe_load(f)

if N_EPOCHS_OVERRIDE is not None:
    config["n_epochs"] = int(N_EPOCHS_OVERRIDE)

config["output_dir"] = str(OUTPUT_DIR)

sections = []
if DATA_DIR is not None and Path(DATA_DIR).exists():
    sections = load_all_sections(Path(DATA_DIR))

if not sections:
    if not USE_SYNTHETIC_IF_MISSING:
        raise RuntimeError("No sections loaded from DATA_DIR and synthetic fallback is disabled.")
    sections = make_synthetic_sections()
    print(f"Using synthetic data: {len(sections)} sections")
else:
    print(f"Using real data: {len(sections)} sections from {Path(DATA_DIR)}")

pd.DataFrame([{"section_id": s["section_id"], "model_type": s["model_type"], "timepoint": s["timepoint"], "cells": len(s["xy_coords"])} for s in sections])


Using synthetic data: 4 sections


,section_id,model_type,timepoint,cells
0,MOG_baseline_synth01,MOG,baseline,180
1,MOG_chronic_long_synth01,MOG,chronic_long,180
2,PLP_remission_I_synth01,PLP,remission_I,180
3,PLP_peak_II_synth01,PLP,peak_II,180


In [6]:
from spatialflow import EAESpatialFlow

pipeline = EAESpatialFlow(config=config, device=DEVICE, output_dir=OUTPUT_DIR)

history = pipeline.train(
    sections,
    n_epochs=config.get("n_epochs", 2),
    lr=config.get("learning_rate", 1e-3),
    warmup_steps=config.get("warmup_steps", 500),
    checkpoint_every=max(1, config.get("n_epochs", 2)),
)

display(pd.DataFrame(history))


2026-02-25 14:49:58,077 [INFO] spatialflow.graph_builder — LRDatabase loaded: 15 pairs (source=fallback)
2026-02-25 14:49:58,101 [INFO] spatialflow.pipeline — Preprocessing 4 sections …
2026-02-25 14:49:58,102 [INFO] spatialflow.aligner — Fitting AlignmentTransform for section: MOG_baseline_synth01
2026-02-25 14:49:58,102 [INFO] spatialflow.aligner — Computing geodesic distance to central canal …
2026-02-25 14:49:58,107 [INFO] spatialflow.aligner — Computing geodesic distance to GM border …
2026-02-25 14:49:58,113 [INFO] spatialflow.aligner — Dorsal unit vector (row, col): (-0.9998034987835144, 0.019823315066936527)
2026-02-25 14:49:58,114 [INFO] spatialflow.graph_builder — Building graph for MOG | MOG_baseline_synth01 | n_cells=360
2026-02-25 14:49:58,321 [INFO] spatialflow.graph_builder — Graph built: 360 nodes, 530088 edges (53208 spatial + 476880 LR)
2026-02-25 14:49:58,323 [INFO] spatialflow.aligner — Fitting AlignmentTransform for section: MOG_chronic_long_synth01
2026-02-25 14:4

,total_loss,elbo_loss,contrastive_loss,divergence_loss,recon_loss,kl_loss
0,17.662966,16.450164,1.827806,0.597797,16.449252,0.263252
1,15.121595,14.004164,1.637576,0.597285,13.999779,0.394009


In [7]:
metrics = pipeline.validate(sections)
results = pipeline.predict(sections)
pipeline.save_outputs(results, sections)

pd.Series(metrics, name="value")


/Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow/spatialflow/velocity.py:311: UserWarning: Inflammatory axis not set — momentum scores will be arbitrary. Call set_inflammatory_axis() before use.
  inf_mom = self.inflammatory_momentum(vel)
2026-02-25 14:50:03,306 [INFO] spatialflow.pipeline — Validation metrics: {'mog_mean_divergence': -3.7858699215576053e-06, 'plp_mean_divergence': 2.503460564184934e-07, 'divergence_separation': 4.036215977976099e-06, 'plp_peak_inf_momentum': 0.0, 'plp_rem_inf_momentum': 0.0, 'momentum_relapse_signal': 0.0}
2026-02-25 14:50:03,629 [INFO] spatialflow.pipeline — Outputs saved to /Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow/outputs/notebook_run
/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
... storing 'section_id' as categorical
... storing 'model_type' as categorical
... 

mog_mean_divergence       -3.785870e-06
plp_mean_divergence        2.503461e-07
divergence_separation      4.036216e-06
plp_peak_inf_momentum      0.000000e+00
plp_rem_inf_momentum       0.000000e+00
momentum_relapse_signal    0.000000e+00
Name: value, dtype: float64

In [8]:
if RUN_ATTRIBUTE:
    remission = [s for s in sections if s["model_type"] == "PLP" and "remission" in s["timepoint"].lower()]
    if remission:
        attr = pipeline.attribute(remission[0], n_top_genes=config.get("n_top_genes", 20))
        display(attr["top_genes"].head(20))
        display(attr["rlcs_attribution"])
    else:
        print("No PLP remission sections found; skipping attribution.")
else:
    print("Attribution disabled.")


/Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow/spatialflow/velocity.py:311: UserWarning: Inflammatory axis not set — momentum scores will be arbitrary. Call set_inflammatory_axis() before use.
  inf_mom = self.inflammatory_momentum(vel)


,gene,attribution_score,rank
0,gene_004,1.781166e-10,1
1,gene_090,1.297175e-10,2
2,gene_026,1.172375e-10,3
3,gene_038,1.159207e-10,4
4,gene_058,1.049801e-10,5
5,tgfbr1,1.003733e-10,6
6,gene_040,8.917417e-11,7
7,gene_075,8.719957e-11,8
8,itgam,8.603512e-11,9
9,gene_069,8.587890e-11,10


,feature,attribution_score
0,d_CC,6.100350e-10
1,d_GM,4.619053e-10
2,theta,5.644568e-11
3,cos_theta,4.795520e-11
4,hemisphere,4.226672e-11
5,sin_theta,3.389697e-11


In [9]:
artifact_files = sorted([p.relative_to(OUTPUT_DIR).as_posix() for p in OUTPUT_DIR.rglob('*') if p.is_file()])
print(f"Saved {len(artifact_files)} files in {OUTPUT_DIR}")
for f in artifact_files:
    print(f" - {f}")

div_path = OUTPUT_DIR / "divergence_scores.csv"
if div_path.exists():
    display(pd.read_csv(div_path).head())


Saved 4 files in /Users/chrislangseth/work/karolinska_institutet/projects/SpatialFlow/outputs/notebook_run
 - checkpoints/spatialflow_epoch0002.pt
 - divergence_scores.csv
 - latent_embeddings.h5ad
 - niche_velocity.parquet


,section_id,model_type,timepoint,cell_idx,divergence,bifurcation,relapse_nursery
0,MOG_baseline_synth01,MOG,baseline,0,0.619129,True,True
1,MOG_baseline_synth01,MOG,baseline,1,-1.953980,False,False
2,MOG_baseline_synth01,MOG,baseline,2,-0.049050,False,False
3,MOG_baseline_synth01,MOG,baseline,3,0.322380,False,False
4,MOG_baseline_synth01,MOG,baseline,4,-0.419450,False,False
